# r/soccer analytical pipeline on Google Colab (GPU)

**FIFA Communities chapter — `code/05-analysis-pipeline.py` run on a Colab GPU.** Orchestration only: every cell calls the
same command-line program that runs locally; nothing analytical lives in this notebook. Design:
`docs/session-04_2026-10-22/01_analytical-pipeline-design.md` (v1.5; §10 for Colab). Records: `docs/session-05_2026-10-23/`
(implementation, dry run), `docs/session-06_2026-10-24/` (the two-step taxonomy protocol D5, superseded), `docs/session-07_2026-10-25/`
(one pipeline, this Drive layout; the runbook `02_runbook-colab-run.md`), `docs/session-08_2026-10-25/` (first Colab run; decisions
D7 — the topic is the unit of analysis, no category layer —, D8 — the topic-density target — and D9 — tiered outlier
reassignment: a second topic for the less confident posts).

**Runtime:** *Runtime → Change runtime type → GPU*, A100 40 GB preferred (L4 acceptable; T4 works in float16, ≈ 3× slower),
high-RAM. In VS Code with the Colab extension: open this notebook, choose *New Colab Server → GPU*.

**Google Drive layout — everything that persists lives in `MyDrive/fifa-communities/`** (decision H2, 2026-10-25). The program
writes there directly through its root setting `PIPELINE_ROOT` (no symlinks); the code runs from a GitHub clone on the Colab
machine, and the 17 CSVs are copied once per session to the machine's local disk for fast reads (decision H3).

| Drive folder (flat) | holds | repository path after the hand-back |
|---|---|---|
| `yearly datasets/` | the 17 cleaned yearly CSVs (input; must exist) | `data/yearly datasets/` |
| `analysis/` | join tables by post id: `period_bins`, `sentiment`, `post_variables` | `data/analysis/` |
| `results/` | `bin-1` … `bin-9/` and the aggregated tables | `results/analysis/` |
| `embeddings/` | sentence-embedding cache per year | `results/embeddings/` (gitignored) |
| *(local disk)* `/content/hf-cache/` | model weights (mpnet, Qwen3-1.7B), re-downloaded per session: safetensors memory-maps them and Drive's mount cannot serve that | `.hf-cache/` (gitignored) |
| `logs/` | one session log per Colab run (cell 0.1) | — |
| `handback/` | zips that rebuild the repository paths (section 5) | — |
| `comparisons/` | optional comparison runs (section 3.2) | — |

**Two passes.** *Pass 1 — smoke test on bin 1* (≈ 15–20 min of GPU time): every stage once, labelling only the 12 largest
topics, with a check cell after each stage that compares with the local dry run (92 topics, 35.9 % raw outliers with MiniLM).
*Pass 2 — the real run*: bin 1 in full (all topics labelled), then the gates and bins 2–9. **The test batch stops after section
3.1** and hands the outputs back with the zip of section 5. Order of operations (design §3): bins → embed + topics → **label** →
*co-author review* (`topic_overrides.csv`: renames, merges) → the outlier tiers of D9 (applied by the topics stage; `reassign` re-applies them) → sentiment →
collect (every post gets its topic's uid, label and tone; the topic table gets the mean VADER score) → hand-back → S6 locally in R.


## 0. Environment (run cells 0.1–0.4 every session; ≈ 3 min)

In [ ]:
# 0.1 Google Drive: mount, check the input folder, create the output folders of the flat layout, open the session log.
import os, sys, datetime
DRIVE = '/content/drive/MyDrive/fifa-communities'     # the shared folder (decision H2): everything that persists lives here
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    raise SystemExit(f'drive.mount failed in this kernel ({e}); open the notebook at colab.research.google.com, or mount Drive by hand')
assert os.path.isdir(f'{DRIVE}/yearly datasets'), f'{DRIVE}/yearly datasets not found on Drive'
n = len([f for f in os.listdir(f'{DRIVE}/yearly datasets') if f.endswith('.csv')])
assert n == 17, f'expected 17 yearly files in {DRIVE}/yearly datasets, found {n}'
for d in ['analysis', 'results', 'embeddings', 'logs', 'handback', 'comparisons']:   # model weights stay on the local disk (cell 0.2)
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
RESULTS, ANALYSIS = f'{DRIVE}/results', f'{DRIVE}/analysis'
# session log on Drive: everything printed through slog() and every stage's output lands here as well
SESSION_LOG = f'{DRIVE}/logs/colab_{datetime.datetime.utcnow():%Y-%m-%d_%H%M}.log'
def slog(*parts):
    line = ' '.join(str(p) for p in parts)
    print(line, flush=True)
    with open(SESSION_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n'); f.flush()
slog(f'=== Colab session log {SESSION_LOG} (UTC {datetime.datetime.utcnow():%Y-%m-%d %H:%M}) ===')
slog('Drive mounted; yearly files:', n, '| folders under fifa-communities:', sorted(os.listdir(DRIVE)))


In [ ]:
# 0.2 Code from GitHub (public repository) on the Colab machine; the 17 CSVs copied once per session to its local disk
#     (Drive I/O is slow for 2.4 GB of CSVs); the program's paths rooted on the Drive folder through PIPELINE_ROOT (H3).
import os, subprocess, shutil, json, sys
REPO = '/content/fifa-reddit-communities'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/alannemi/fifa-reddit-communities', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
os.chdir(REPO)
slog('repository at commit', subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())
LOCAL_YEARLY = '/content/data/yearly datasets'
if not os.path.isdir(LOCAL_YEARLY):
    shutil.copytree(f'{DRIVE}/yearly datasets', LOCAL_YEARLY)
files = sorted(f for f in os.listdir(LOCAL_YEARLY) if f.endswith('.csv'))
assert len(files) == 17, files
slog('yearly files on the local disk:', len(files), '| total GB:', round(sum(os.path.getsize(f'{LOCAL_YEARLY}/{f}') for f in files) / 1e9, 2))
os.environ['PIPELINE_ROOT'] = DRIVE                 # analysis/, results/, embeddings/ under the Drive folder
os.environ['PIPELINE_YEARLY_DIR'] = LOCAL_YEARLY    # the input alone is read from the local copy
os.environ['PIPELINE_HF_CACHE_DIR'] = '/content/hf-cache'   # model weights on the local disk, re-downloaded per session (≈ 4 GB,
#   minutes): safetensors memory-maps the weights and Drive's FUSE mount returns garbage for such reads ("header too large",
#   first Colab run 2026-10-25); the Drive folder hf-cache/ is therefore unused
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
probe = ('import json, sys; sys.path.insert(0, "code"); from topicmodel import config as C; '
         'print(json.dumps({k: str(getattr(C, k)) for k in ["PIPELINE_ROOT", "YEARLY_DIR", "ANALYSIS_DATA_DIR", "ANALYSIS_DIR", "EMBED_CACHE_DIR", "LLM_CACHE_DIR"]}))')
paths = json.loads(subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True, check=True, env=os.environ).stdout)
slog('effective paths of the program:', json.dumps(paths, indent=1))
assert paths['ANALYSIS_DIR'] == RESULTS and paths['ANALYSIS_DATA_DIR'] == ANALYSIS and paths['YEARLY_DIR'] == LOCAL_YEARLY and paths['LLM_CACHE_DIR'] == '/content/hf-cache', paths


In [ ]:
# 0.3 Environment: install ONLY the pinned packages on top of Colab's RAPIDS stack (review finding R8), then prove that
#     cuML, CUDA and the pins are intact. If `pip check` reports a broken requirement, fix that one package explicitly
#     (do not re-run a plain `pip install -r`). Everything is written to the session log.
import subprocess, sys
inst = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-r', 'code/requirements-colab.txt'], capture_output=True, text=True)
slog('pip install (--no-deps) exit', inst.returncode, (inst.stdout + inst.stderr).strip()[-2000:])
assert inst.returncode == 0, 'pip install failed — read the output above'
chk = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
slog('pip check:', chk.stdout.strip() or 'OK')
# Only complaints that name the pipeline's stack matter; Colab's own image has known gaps (e.g. "ipython requires jedi")
# that have nothing to do with the packages installed above and can be ignored or fixed with `pip install jedi`.
OUR_STACK = ('bertopic', 'umap', 'hdbscan', 'pynndescent', 'sentence-transformers', 'sentence_transformers', 'vadersentiment',
             'transformers', 'torch', 'numpy', 'scikit-learn', 'scikit_learn', 'pandas', 'scipy', 'cuml', 'numba', 'huggingface')
ours = [l for l in chk.stdout.splitlines() if any(k in l.lower() for k in OUR_STACK)]
slog('pip check (pipeline stack):', ' | '.join(ours) if ours else 'OK — other complaints above concern the Colab image only')
assert not ours, 'pip check names a package of the pipeline stack — install that one package explicitly and re-run this cell'
import torch, cuml, bertopic, umap, hdbscan, sentence_transformers, transformers, numpy, sklearn, pandas   # every import must work
from importlib.metadata import version as dist_version        # versions from the installed distributions: not every module
pins = {d: dist_version(d) for d in ['bertopic', 'umap-learn', 'hdbscan', 'pynndescent', 'sentence-transformers',   # exposes __version__
                                      'transformers', 'torch', 'numpy', 'scikit-learn', 'pandas', 'vaderSentiment']}   # (hdbscan does not)
assert torch.cuda.is_available(), 'no CUDA device — change the runtime type to GPU'
slog('GPU:', torch.cuda.get_device_name(0), '| torch', pins['torch'], '| CUDA', torch.version.cuda, '| cuml', cuml.__version__, '| python', sys.version.split()[0])
slog('versions:', ' | '.join(f'{k} {v}' for k, v in pins.items()))
assert pins['bertopic'] == '0.17.4' and pins['umap-learn'] == '0.5.12' and pins['hdbscan'] == '0.8.44', 'pins not honoured'
try:
    from cuml.accel import enabled as _accel_enabled
    slog('cuml.accel enabled in this kernel:', _accel_enabled(), '(the topics stage runs in a child process under `python -m cuml.accel`)')
except Exception as e:
    slog('cuml.accel state not readable:', e)
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv'], capture_output=True, text=True)
slog(smi.stdout.strip())


In [ ]:
# 0.4 Helpers. stage() runs one stage of the program (accel=True prefixes `python -m cuml.accel` for the GPU UMAP/HDBSCAN),
#     streams its output line by line to the notebook AND to the session log (child unbuffered, progress bars off;
#     review finding R33) and raises on a non-zero exit. The environment (PIPELINE_ROOT etc.) is taken from os.environ at
#     every call, so re-running cell 0.2 takes effect. cfg() reads a bin's config.json on Drive. report_ok() asserts that a
#     bin's report.html exists: the program renders it after topics/reassign/label but never fails a stage over it
#     (presentation only), so a skipped report shows up only as a WARNING in results/bin-N/log.txt — on the first Colab run
#     (2026-10-25) Pass 1 ended without a report because of a plotly 5.x defect, unnoticed until the file was looked for.
import subprocess, sys, shlex, time, json, os
from collections import Counter
import pandas as pd
def run_logged(cmd):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'HF_HUB_DISABLE_PROGRESS_BARS': '1', 'TQDM_DISABLE': '1'}
    slog('>>>', ' '.join(cmd)); t0 = time.time()
    with open(SESSION_LOG, 'a', encoding='utf-8') as f, subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                                           text=True, env=env, bufsize=1) as p:
        for line in p.stdout:
            print(line, end='', flush=True); f.write(line); f.flush()
    slog(f'<<< exit {p.returncode} after {(time.time() - t0) / 60:.1f} min')
    if p.returncode != 0:
        raise RuntimeError(f'command failed (exit {p.returncode}): {" ".join(cmd)}')
def stage(args, accel=False):
    run_logged([sys.executable, '-u'] + (['-m', 'cuml.accel'] if accel else []) + ['code/05-analysis-pipeline.py'] + shlex.split(args))
def cfg(b):
    return json.load(open(f'{RESULTS}/bin-{b}/config.json'))
OUTLIER_GATE = 0.50     # raw outlier share a bin may have when its guard converged (design §12.2 said 0.45; 0.50 proposed 2026-10-25 after
                        # bin 3's 45.8 %, pending the author's confirmation — under D9 the share only sizes the dual and outlier tiers)
def floor_bound(c):
    """Decision D10 (2026-10-25): the guard stopped at the floor min_cluster_size with too FEW topics — it cannot go below the floor
    (design §6.1), so the floor wins over the density target and the bin stands. The pipeline records guard.floor_bound since D10;
    for a bin clustered before (bin 7) the state is derived from the recorded trial."""
    g = c['guard']
    if 'floor_bound' in g:
        return bool(g['floor_bound'])
    floor = c.get('hdbscan', {}).get('min_cluster_floor', 25)
    return (not g['converged']) and g['min_cluster_size'] == floor and g['n_topics'] < g['target'][0]
def gate_ok(c):
    """The large-bin gate on a bin's topics section of config.json (design §12.2): raw outliers within OUTLIER_GATE and a guard
    that converged to the density target or is floor-bound (D10)."""
    return c['outlier_share_raw'] <= OUTLIER_GATE and (c['guard']['converged'] or floor_bound(c))
def review_pack(b):
    """review_pack.md of a bin on Drive (code/08-review-pack.py), right after its labelling — what the co-authors read."""
    run_logged([sys.executable, '-u', 'code/08-review-pack.py', 'pack', '--bin', str(b), '--analysis-dir', RESULTS])
def status():
    """Where every bin stands (status.json per bin; REVIEW_READY.txt appears when a bin's topics are all labelled)."""
    run_logged([sys.executable, '-u', 'code/05-analysis-pipeline.py', '--stage', 'status'])
def handback(bins):
    """Zip the bins' outputs (without model/), the aggregated tables, analysis/* and the session log as repository paths into
    MyDrive/fifa-communities/handback/ — unzips at the repository root. Called per bin by section 3.5 and by section 5."""
    import zipfile, datetime
    stamp = datetime.datetime.utcnow().strftime('%Y-%m-%d_%H%M')
    zpath = f"{DRIVE}/handback/bins-{'-'.join(map(str, bins))}_{stamp}.zip"
    n_files = 0
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for b in bins:
            for dirpath, dirs, files in os.walk(f'{RESULTS}/bin-{b}'):
                dirs[:] = [d for d in dirs if d != 'model']
                for fn in files:
                    p = os.path.join(dirpath, fn)
                    z.write(p, arcname=os.path.join('results/analysis', os.path.relpath(p, RESULTS))); n_files += 1
        for fn in os.listdir(RESULTS):                                    # the aggregated tables at the top level
            if fn.endswith('.csv'):
                z.write(f'{RESULTS}/{fn}', arcname=f'results/analysis/{fn}'); n_files += 1
        for fn in os.listdir(ANALYSIS):
            if fn.endswith(('.csv', '.json', '.txt')):
                z.write(f'{ANALYSIS}/{fn}', arcname=f'data/analysis/{fn}'); n_files += 1
        z.write(SESSION_LOG, arcname='logs/' + os.path.basename(SESSION_LOG)); n_files += 1
    slog('hand-back zip:', zpath, '|', n_files, 'files |', round(os.path.getsize(zpath) / 1e6, 1), 'MB')
    return zpath
def report_ok(b):
    p = f'{RESULTS}/bin-{b}/report.html'
    assert os.path.exists(p), f'{p} missing — look for "report for bin {b} skipped" in {RESULTS}/bin-{b}/log.txt, fix, then stage("--stage report --bin {b}")'
    slog('report:', p)
def unit_checks():
    run_logged([sys.executable, '-u', 'code/tests/test_pipeline_units.py'])
unit_checks()          # ~1 min, no model needed — proves the code imports and behaves on this machine


## 1. S0 — period bins (every post; Σ must equal 1,302,021 and the nine counts the design table)

In [ ]:
stage('--stage bins')
s = pd.read_csv(f'{ANALYSIS}/period_bins_summary.csv')
slog(s[['period_bin', 'period_type', 'n_posts', 'expected_n_design', 'target_topics_min', 'target_topics_max']].to_string(index=False))
assert (s.n_posts == s.expected_n_design).all() and s.n_posts.sum() == 1302021, 'bin sizes differ from the design table'


## 2. Pass 1 — smoke test on bin 1 (≈ 15–20 min of GPU time)
Every stage once on bin 1, with labelling limited to the 12 largest topics. Each check cell compares with the local dry run
of 2026-10-23 (MiniLM, CPU): 10,002 posts, 9,809 modelled, **92 topics at min_cluster_size 25, 35.9 % raw outliers**;
12 labelled topics, 1 needing review. The first Colab run (mpnet, 2026-10-25) gave 65 topics and 34.8 %. The guard's target
is the density of decision D8: 50–75 topics per 10,000 modelled posts (bin 1: 49–74). Since decision D7 the topic is the
unit of analysis — there is no taxonomy stage. Outputs go to the real `results/bin-1/` on Drive; Pass 2 resumes from them
(labelling continues from `labels.jsonl`).


In [ ]:
stage('--stage embed --bin 1')
stage('--stage topics --bin 1', accel=True)
c = cfg(1)['topics']
slog(json.dumps({k: c[k] for k in ['n_documents', 'n_modelled', 'n_topics_raw', 'outlier_share_raw', 'elapsed_min']}, indent=1))
slog('backend:', c['backend']); slog('guard:', c['guard'])
assert c['backend']['backend'] == 'cuml', 'UMAP/HDBSCAN did not run on cuML — check that the command was prefixed with python -m cuml.accel'
assert c['guard']['converged'], 'topic-density guard did not reach the target (D8: 50–75 topics per 10k posts): see guard trials above'
slog('calibration candidates (T, n reassigned):', [(r['threshold'], r['n_reassigned_at_T']) for r in c['calibration']])
report_ok(1)


In [ ]:
stage('--stage label --bin 1 --max-topics 12')
t = pd.read_csv(f'{RESULTS}/bin-1/topics_labelled.csv')
slog(t[['topic_id', 'name', 'tone_category', 'round_iterations', 'agreement_mode', 'needs_review']].to_string(index=False))
l = cfg(1)['label']; slog('llm:', l['llm_model'], l['llm_revision'], l['dtype'], 'batch', l['batch_size'], '| prompt', l['prompt_version'], '| stats', l['llm_stats'], '| min', l['elapsed_min'])
assert len(t) == 12 and l['llm_stats']['truncated'] == 0


In [ ]:
stage('--stage sentiment --years 2010')      # the 2010 file only, for the smoke test; Pass 2 scores every year
stage('--stage collect --allow-partial')
side = json.load(open(f'{ANALYSIS}/post_variables.json'))
slog(json.dumps({k: side[k] for k in ['n_posts', 'bins_complete', 'complete', 'n_with_sentiment', 'n_with_topic', 'n_topics_labelled', 'errors']}, indent=1))
assert side['n_posts'] == 1302021 and side['bins_complete'] == [1] and not side['errors']
report_ok(1)
slog(f'Pass 1 complete — open {RESULTS}/bin-1/report.html (download it from Drive, or render it here with IPython.display.HTML) and compare with the local dry run.')


## 3. Pass 2 — the real run
### 3.1 Bin 1 in full — gate 1 (design §12.1) and gate 4 (§12.4) — **the test batch ends here**
Resumes from Pass 1: the remaining topics are labelled (the unit of analysis since decision D7). Read `report.html` (2010
themes: vuvuzelas, Jabulani, Lampard's goal, Suárez's handball, Spain's win, Paul the octopus) and the label summary
(`needs_review` ≤ 15 %, the `agreement_mode` and round distributions).


In [ ]:
stage('--stage label --bin 1')                 # resumable; re-run the cell after a disconnect
t = pd.read_csv(f'{RESULTS}/bin-1/topics_labelled.csv')
slog(f'{len(t)} topics | needs_review {t.needs_review.sum()} ({t.needs_review.mean():.0%}) | modes {t.agreement_mode.value_counts().to_dict()} | rounds {t.round_iterations.value_counts().sort_index().to_dict()}')
if t.needs_review.mean() > 0.15:
    slog('WARNING: needs_review above the 15 % gate (design §12.4) — a decision for the author before bins 2–9')
report_ok(1)
review_pack(1)
stage('--stage collect --allow-partial')
slog('=== STOP POINT of the test batch: run the hand-back cell in section 5 ===')


### Stop point of the test batch
Everything above is the agreed test batch (Pass 1 + the full bin 1). Run the **hand-back cell in section 5**, which zips
`results/bin-1/` (without the regenerable `model/`), the aggregated tables, `analysis/` and the session log — laid out as
repository paths — into `MyDrive/fifa-communities/handback/`. Download the zip and unzip it at the repository root, or let the
reviewer read it from Drive. The cells below (comparisons, large-bin gate, `T`, bins 2–9, sentiment for all years, the complete
collect) belong to the next session.


### 3.2 Optional comparison on bin 1 (design §12.4, F17: seeded sampling) — into `comparisons/`, so bin 1's real outputs stay clean
The Qwen3-4B comparison (design F9) was dropped by the author on 2026-10-24: the pipeline stays with Qwen3-1.7B. A comparison
run uses `--out-root`, which keeps the repository-shaped sub-layout (`results/analysis/`, `data/analysis/`) inside its folder.


In [ ]:
RUN_COMPARISONS = False
if RUN_COMPARISONS:
    import shutil
    for name, extra in [('cmp-seeded', '--sampling seeded')]:
        root = f'{DRIVE}/comparisons/{name}'
        os.makedirs(f'{root}/results/analysis', exist_ok=True); os.makedirs(f'{root}/data/analysis', exist_ok=True)
        if not os.path.isdir(f'{root}/results/analysis/bin-1'):
            shutil.copytree(f'{RESULTS}/bin-1', f'{root}/results/analysis/bin-1', ignore=shutil.ignore_patterns('labels*', 'topics_labelled.csv', 'topic_overrides.csv', 'review_pack.md'))
            shutil.copy(f'{ANALYSIS}/period_bins.csv', f'{root}/data/analysis/')
        stage(f'--stage label --bin 1 {extra} --out-root {root}')
        t2 = pd.read_csv(f'{root}/results/analysis/bin-1/topics_labelled.csv')
        slog(name, '| needs_review', t2.needs_review.mean().round(3), '| modes', t2.agreement_mode.value_counts().to_dict())


### 3.3 Large-bin gate (design §12.2): bin 4 (331k posts) or bin 6 (306k)
Raw outlier share ≤ `OUTLIER_GATE` (cell 0.4; 0.45 in the design, 0.50 proposed after bin 3), guard converged (density target of D8) within six trials, minutes on the GPU. If the gate fails — as it did on 2026-10-25 (74 topics,
56 % raw outliers) — cell 3.3b runs the HDBSCAN grid on the cached UMAP array, the author decides the setting (design §14), and
cell 3.3c applies it and continues.


In [ ]:
GATE_BIN = 4
stage(f'--stage embed --bin {GATE_BIN}')
stage(f'--stage topics --bin {GATE_BIN}', accel=True)
c = cfg(GATE_BIN)['topics']
slog(json.dumps({k: c[k] for k in ['n_modelled', 'n_topics_raw', 'outlier_share_raw', 'elapsed_min']}, indent=1), '| guard', c['guard'], '| backend', c['backend']['backend'])
assert gate_ok(c), f'large-bin gate failed (outliers {c["outlier_share_raw"]:.1%} > {OUTLIER_GATE:.0%} or guard neither converged nor floor-bound) — see design §12.2'
stage(f'--stage label --bin {GATE_BIN}')
report_ok(GATE_BIN)


### 3.3b The gate failed: HDBSCAN grid on the cached UMAP array of the gate bin (design §12.2 diagnostic)
`--stage grid` refits HDBSCAN over `config.HDBSCAN_GRID` (min_cluster_size × min_samples × selection method × epsilon) on `bin-N/umap_5d.npy` — minutes on the GPU, nothing else in the bin changes — and writes
`bin-N/hdbscan_grid.csv`. The author chooses the setting from this table: `HDBSCAN_MIN_SAMPLES`, `HDBSCAN_SELECTION_METHOD`,
`HDBSCAN_EPSILON`, each either one value for every bin or a dict by period type; `in_range` refers to the density target of D8 (`{"tournament": …, "interim": …}`).


In [ ]:
stage(f'--stage grid --bin {GATE_BIN}', accel=True)
g = pd.read_csv(f'{RESULTS}/bin-{GATE_BIN}/hdbscan_grid.csv')
slog(g.sort_values('outlier_share').to_string(index=False))
ok = g[g.in_range & (g.outlier_share <= OUTLIER_GATE)]
slog(f'{len(ok)} setting(s) inside the density target with <= {OUTLIER_GATE:.0%} raw outliers' + (':' if len(ok) else ''))
if len(ok): slog(ok[['min_cluster_size', 'min_samples', 'method', 'epsilon', 'n_topics', 'outlier_share', 'largest_share']].to_string(index=False))


### 3.3c After the decision: refit HDBSCAN with the new setting, re-check the gate, continue
The decision goes into `config.py` (and the design's decisions log), is committed, and cell 0.2 pulls it. `--force --keep-umap` keeps
`umap_5d.npy` and refits HDBSCAN with the guard under the new setting; the gate checks are the same as in cell 3.3; then the labels. If the setting
applies to every bin (not per period type), bin 1 has to be re-clustered and relabelled too before its review starts.


In [ ]:
stage(f'--stage topics --bin {GATE_BIN} --force --keep-umap', accel=True)
c = cfg(GATE_BIN)['topics']
slog(json.dumps({k: c[k] for k in ['n_modelled', 'n_topics_raw', 'outlier_share_raw', 'elapsed_min']}, indent=1), '| hdbscan', c.get('hdbscan'), '| guard', c['guard'])
assert gate_ok(c), 'large-bin gate failed again — see design §12.2'
stage(f'--stage label --bin {GATE_BIN}')
report_ok(GATE_BIN)
review_pack(GATE_BIN)
handback([GATE_BIN])
status()


### 3.4 Outlier tiers (decision D9, 2026-10-25) for bins clustered before it
Since D9 the topics stage applies the tiers itself: an outlier whose similarity to its nearest topic reaches the bin's **35th
percentile** joins that topic; below the **80th percentile** it also carries the second-nearest topic as a second label
(`topic_assignment = dual`; counts and shares use the first topic only); below the 35th percentile it stays an outlier. Bins
clustered before D9 (1 and 3) get the tiers through `--stage reassign` (seconds per bin, no refit; `topic_id_raw`, keywords and
labels never change). The same loop re-applies the tiers to every bin if a quantile changes in `config.py`
(`OUTLIER_REASSIGN_QUANTILE`, `OUTLIER_SINGLE_QUANTILE`) after the co-authors have coded the samples in `outlier_calibration/`.


In [ ]:
# the tiers for every modelled bin whose config.json shows none yet (clustered before D9); after a quantile change: FORCE_ALL = True
FORCE_ALL = False
for b in range(1, 10):
    p = f'{RESULTS}/bin-{b}/config.json'
    if not os.path.exists(p):
        continue
    c = cfg(b)
    if FORCE_ALL or ('topics' in c and c['topics'].get('outlier_threshold') is None and 'reassign' not in c):
        stage(f'--stage reassign --bin {b}')
        r = cfg(b)['reassign']
        slog(f"bin {b}: tiers T={r['outlier_threshold']} C={r['single_threshold']} -> {r['n_reassigned']} reassigned, {r['n_dual']} dual, {r['outlier_share_final']:.1%} outliers left")
status()


### 3.5 Bins 2–9, one bin at a time — the tournament bins (3, 5, 7, 9) first, then the interim bins (2, 4, 6, 8)
embed + topics → gate → label → review pack → hand-back zip → **REVIEW_READY flag**. Each bin is finished before the next one starts, so the co-authors can review bin by bin while the GPU works on. For every
bin the loop (1) embeds and clusters it (a bin clustered before decision D8 is refitted on its cached UMAP array), (2) checks the
gate (raw outliers ≤ `OUTLIER_GATE`, guard converged to the density target) and **stops at the first bin that fails** — run the grid of
cell 3.3b for that bin, decide the setting, re-run this cell: finished bins are skipped —, (3) labels every topic (resumable),
(4) writes `review_pack.md` and `REVIEW_READY.txt` into `results/bin-N/` on Drive and a per-bin hand-back zip into `handback/`,
and logs **"bin N READY FOR REVIEW"**. `status()` (cell 3.6) prints where every bin stands. The outlier tiers of D9 are applied by the
topics stage; a bin clustered before D9 gets them through `reassign` inside the loop.

In [ ]:
BINS = [3, 5, 7, 9, 2, 4, 6, 8]          # the tournament bins first, then the interim bins (author, 2026-10-25); finished stages are
                                          # skipped, so the cell can be re-run after a stop or a disconnect
for b in BINS:
    bdir = f'{RESULTS}/bin-{b}'
    stage(f'--stage embed --bin {b}')
    pre_d8 = os.path.exists(f'{bdir}/documents.csv') and 'hdbscan' not in cfg(b).get('topics', {})   # clustered before decision D8
    stage(f'--stage topics --bin {b}' + (' --force --keep-umap' if pre_d8 else ''), accel=True)
    c = cfg(b)['topics']
    slog(f"bin {b}: {c['n_modelled']} posts, {c['n_topics_raw']} topics (target {c['guard'].get('target', '?')}), raw outliers {c['outlier_share_raw']:.1%}, "
         f"backend {c['backend']['backend']}, guard converged {c['guard']['converged']}, floor-bound {floor_bound(c)}, hdbscan {c.get('hdbscan')}")
    assert c['backend']['backend'] == 'cuml', f'bin {b}: UMAP/HDBSCAN did not run on cuML'
    assert gate_ok(c), \
        f'bin {b}: gate failed (design §12.2: outliers {c["outlier_share_raw"]:.1%} > {OUTLIER_GATE:.0%} or guard neither converged nor floor-bound) — run the grid (cell 3.3b with GATE_BIN = {b}), decide the setting, re-run this cell'
    if c.get('outlier_threshold') is None and 'reassign' not in cfg(b):   # clustered before decision D9: apply the tiers (seconds, no refit)
        stage(f'--stage reassign --bin {b}')
    r = cfg(b).get('reassign') or cfg(b)['topics']
    slog(f"bin {b}: tiers T={r['outlier_threshold']} C={r['single_threshold']} -> {r['n_reassigned']} reassigned, {r['n_dual']} dual, {r['outlier_share_final']:.1%} outliers left")
    report_ok(b)
    stage(f'--stage label --bin {b}')                 # resumable; re-run the cell after a disconnect
    t = pd.read_csv(f'{bdir}/topics_labelled.csv')
    slog(f"bin {b}: {len(t)} topics labelled | needs_review {t.needs_review.sum()} ({t.needs_review.mean():.0%}) | modes {t.agreement_mode.value_counts().to_dict()}")
    report_ok(b)
    review_pack(b)
    handback([b])
    slog(f'=== bin {b} READY FOR REVIEW: {bdir}/REVIEW_READY.txt, review_pack.md and topic_overrides.csv on Drive; the zip in handback/ ===')
status()


### 3.6 Where every bin stands
One row per bin from `status.json`: state, modelled posts, topics, labelled topics, `review_ready` (= `REVIEW_READY.txt` on Drive), the tiers
in force (`outlier_threshold`, `single_threshold`, `n_dual`). Run it any time.


In [ ]:
status()


## 4. S4 sentiment (every year, CPU, ≈ 15 min) and S5 collect

In [ ]:
stage('--stage sentiment --force')
stage('--stage collect')          # without --allow-partial: must end with complete = true
side = json.load(open(f'{ANALYSIS}/post_variables.json'))
slog(json.dumps({k: v for k, v in side.items() if k != 'versions'}, indent=1))
assert side['complete'], 'collection not complete — see bins_state above'


## 5. Hand-back, copy back and finish locally
Everything is already on Drive (`results/`, `analysis/`). Section 3.5 writes one zip per bin as soon as its labelling is done; the
cell below zips a chosen set of bins — what the reviewer and the repository need —
`results/bin-N/` without `model/`, the aggregated tables, `analysis/*.csv|json|txt`, the session log — **laid out as repository
paths** (`results/analysis/…`, `data/analysis/…`, `logs/…`) into `MyDrive/fifa-communities/handback/`. Download the zip and unzip
it at the repository root (`results/analysis/` is tracked in git; `data/analysis/` goes to the shared data folder). After the
full run, stage S6 runs locally:
```bash
Rscript code/06-tag-corpus.R && Rscript code/07-validate-analysis.R
```
and codebook v3.0 is finalised (`docs/session-07_2026-10-25/02_runbook-colab-run.md` §11).


In [ ]:
# hand-back zip of the chosen bins (the per-bin zips of section 3.5 are produced automatically; this cell is for a chosen set)
HANDBACK_BINS = [1]
handback(HANDBACK_BINS)
status()
